In [1]:
%cd /media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject
from __future__ import annotations

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject


In [2]:
# =============================================================================
# IMPORTS
# =============================================================================
 
import logging
import traceback
from typing import Any, Dict, List, Optional
 
import pandas as pd
from langchain_core.documents import Document
from neo4j import GraphDatabase
 
from src.Chunking import get_chunks
from src.Config import get_settings
from src.Utils import (
    _stable_id,
    Amendment,
    Schedule,
    ScheduleEntry,
    norm_regu,
    LawExtractor,
    AmendmentExtractor
)
 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =============================================================================
# CHUNK-BASED INGESTION
# =============================================================================
 
def ingest_dataset(chunks: List[Document]) -> List[Dict]:
    law_data: Dict[str, Dict] = {}
    for chunk in chunks:
        if chunk.metadata.get("chunk_type") != "article":
            continue
        law_key = chunk.metadata["law_id"]
        if law_key not in law_data:
            law_data[law_key] = {
                "law_title": chunk.metadata.get("law_title", law_key),
                "articles":  [],
            }
        law_data[law_key]["articles"].append({
            "article_number": chunk.metadata.get("article_number"),
            "text":           chunk.page_content,
            "language":       "ar",
        })
 
    summaries = []
    for law_key, data in law_data.items():
        articles = data["articles"]
        try:
            if not articles:
                summaries.append({"law_key": law_key, "articles": [], "error": "empty"})
                continue
            ext = LawExtractor(law_key, "\n".join(a["text"] for a in articles))
            summaries.append({
                "law_key":     law_key,
                "law_meta":    ext.extract().law_meta,
                "articles":    articles,
                "penalties":   ext._penalties(articles),
                "definitions": ext._definitions(articles),
                "references":  ext._references(articles),
                "topics":      ext._topics(articles),
                "schedules":   [],   # always empty — tables live in separate chunks
                "error":       None,
            })
        except Exception as e:
            logger.error(f"  [ERROR] {law_key}: {e}")
            summaries.append({"law_key": law_key, "articles": [], "error": str(e)})
    return summaries
 
 
def ingest_amendments(chunks: List[Document]) -> Dict[str, List[Amendment]]:
    groups: Dict[tuple, Dict] = {}
    for chunk in chunks:
        if chunk.metadata.get("chunk_type") != "amended_article":
            continue
        m       = chunk.metadata
        law_key = m["law_id"]
        law_num = m.get("amendment_law_number", "unknown")
        adate   = m.get("amendment_date",       "unknown")
        key     = (law_key, law_num, adate)
        if key not in groups:
            groups[key] = {"law_key": law_key, "texts": [], "articles": []}
        if (no := m.get("article_number")) and no not in groups[key]["articles"]:
            groups[key]["articles"].append(no)
        groups[key]["texts"].append(chunk.page_content)
 
    result: Dict[str, List[Amendment]] = {}
    for (law_key, _, _), data in groups.items():
        full_text = "\n".join(data["texts"])
        ext       = AmendmentExtractor(full_text)
        amendment = ext.extract(target_law_id=law_key)
        if amendment:
            amendment.amended_article_numbers = data["articles"] or amendment.amended_article_numbers
            result.setdefault(law_key, []).append(amendment)
    return result
 
 
def ingest_tables(chunks: List[Document]) -> Dict[str, List[Schedule]]:
    """
    Parse all chunks with chunk_type == "table" and return
    { law_id -> [Schedule, ...] } using LawExtractor._extract_schedules().
 
    The schedule type ("drug" / "weapon") is looked up from
    norm_regu._LAW_REGISTRY so the correct entry parser is chosen automatically.
    Table chunks for the same law are concatenated in order before parsing.
    """
    # ── group table chunks by law_id ──────────────────────────────────────
    law_texts: Dict[str, str] = {}
    for chunk in chunks:
        if chunk.metadata.get("chunk_type") != "table":
            continue
        law_id = chunk.metadata.get("law_id")
        if not law_id:
            logger.warning("table chunk missing law_id — skipped")
            continue
        law_texts[law_id] = law_texts.get(law_id, "") + "\n" + chunk.page_content
 
    # ── extract schedules via existing LawExtractor machinery ─────────────
    result: Dict[str, List[Schedule]] = {}
    for law_id, text in law_texts.items():
        if law_id not in norm_regu._LAW_REGISTRY.value:
            logger.warning(f"[ingest_tables] Unknown law_id '{law_id}' — skipped")
            continue
        try:
            schedules = LawExtractor(law_id, text)._extract_schedules()
            if schedules:
                result[law_id] = schedules
                logger.info(
                    f"  ✓ {law_id}: {len(schedules)} table(s) | "
                    + " | ".join(
                        f"table {s.schedule_number} ({len(s.entries)} entries)"
                        for s in schedules
                    )
                )
            else:
                logger.warning(f"  [ingest_tables] {law_id}: no tables extracted")
        except Exception as e:
            logger.error(f"  [ingest_tables] {law_id}: {e}")
 
    return result

In [4]:
# =============================================================================
# SECTION 5: KNOWLEDGE GRAPH
# =============================================================================
 
class LegalKnowledgeGraph:
    """
    Neo4j knowledge graph for Egyptian criminal law.
 
    All Cypher goes through ``_run()`` — one session per call, auto-closed.
    Schema and statistics lists are read from ``norm_regu`` — single source
    of truth, no duplication.
    """
 
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()
        logger.info("✓ Neo4j connected")
 
    def close(self):
        self.driver.close()
 
    def _run(self, query: str, **params):
        with self.driver.session() as s:
            s.run(query, **params)
 
    # ── schema ────────────────────────────────────────────────────────────
 
    def setup_schema(self, drop_existing: bool = False):
        with self.driver.session() as s:
            if drop_existing:
                s.run("MATCH (n) DETACH DELETE n")
                logger.info("✓ Existing data cleared")
            for stmt in norm_regu.CREATION_OF_SCHEMA.value:
                try:
                    s.run(stmt)
                except Exception:
                    pass
        logger.info("✓ Schema ready")
 
    # ── node / relationship creators ──────────────────────────────────────
 
    def create_law(self, meta: Dict[str, Any]):
        self._run("""
            MERGE (l:Law {law_id: $law_id})
            SET l.title             = $title,
                l.promulgation_date = $promulgation_date,
                l.source            = $source,
                l.language          = $language
        """, law_id=meta["law_id"], title=meta.get("title"),
            promulgation_date=meta.get("promulgation_date"),
            source=meta.get("source"), language=meta.get("language", "ar"))
 
    def create_article(self, law_id: str, article_number: Optional[str],
                       text: str, position: int):
        self._run("""
            MATCH (l:Law {law_id: $law_id})
            MERGE (a:Article {article_id: $article_id})
            SET a.article_number = $article_number,
                a.text           = $text,
                a.law_id         = $law_id
            MERGE (l)-[r:CONTAINS]->(a)
            SET r.position = $position
        """, law_id=law_id,
            article_id=_stable_id(law_id, article_number or "preamble", position),
            article_number=article_number, text=text, position=position)
 
    def create_penalty(self, penalty: Dict, law_id: str):
        self._run("""
            MATCH (a:Article {law_id: $law_id, article_number: $article_number})
            MERGE (p:Penalty {penalty_id: $penalty_id})
            SET p.penalty_type = $penalty_type,
                p.min_value    = $min_value,
                p.max_value    = $max_value,
                p.unit         = $unit
            MERGE (a)-[:HAS_PENALTY]->(p)
        """, law_id=law_id,
            penalty_id=_stable_id(law_id, penalty.get("article_number"), penalty.get("penalty_type")),
            article_number=penalty.get("article_number"),
            penalty_type=penalty.get("penalty_type"),
            min_value=penalty.get("min_value"),
            max_value=penalty.get("max_value"),
            unit=penalty.get("unit"))
 
    def create_definition(self, definition: Dict, law_id: str):
        self._run("""
            MATCH (a:Article {law_id: $law_id, article_number: $article_number})
            MERGE (d:Definition {definition_id: $def_id})
            SET d.term            = $term,
                d.definition_text = $definition_text
            MERGE (a)-[:DEFINES]->(d)
        """, law_id=law_id,
            def_id=_stable_id(law_id, definition.get("term")),
            article_number=definition.get("defined_in_article"),
            term=definition.get("term"),
            definition_text=definition.get("definition_text"))
 
    def create_topic(self, topic_name: str):
        self._run("MERGE (t:Topic {topic_id: $id, name: $name})",
                  id=_stable_id(topic_name), name=topic_name)
 
    def link_article_topic(self, law_id: str, article_number: str,
                           topic_name: str, confidence: float):
        self._run("""
            MATCH (a:Article {law_id: $law_id, article_number: $article_number})
            MATCH (t:Topic {name: $topic_name})
            MERGE (a)-[r:TAGGED_WITH]->(t)
            SET r.confidence = $confidence
        """, law_id=law_id, article_number=article_number,
            topic_name=topic_name, confidence=confidence)
 
    def create_reference(self, law_id: str, from_article: str, to_article: str):
        self._run("""
            MATCH (from:Article {law_id: $law_id, article_number: $from_article})
            MATCH (to:Article   {law_id: $law_id, article_number: $to_article})
            MERGE (from)-[r:REFERENCES]->(to)
            SET r.reference_type = 'direct'
        """, law_id=law_id, from_article=from_article, to_article=to_article)
 
    def create_schedule(self, law_id: str, schedule: Schedule):
        self._run("""
            MATCH (l:Law {law_id: $law_id})
            MERGE (s:Schedule {schedule_id: $schedule_id})
            SET s.schedule_number = $num,
                s.title           = $title,
                s.entry_count     = $count
            MERGE (l)-[:HAS_SCHEDULE]->(s)
        """, law_id=law_id, schedule_id=schedule.schedule_id,
            num=schedule.schedule_number, title=schedule.title,
            count=len(schedule.entries))
 
    def create_schedule_entry(self, schedule_id: str, entry: ScheduleEntry, position: int):
        entry_id = _stable_id(schedule_id, entry.entry_number, position)
        self._run("""
            MATCH (s:Schedule {schedule_id: $schedule_id})
            MERGE (e:ScheduleEntry {entry_id: $entry_id})
            SET e.entry_number = $entry_number,
                e.arabic_name  = $arabic_name,
                e.english_name = $english_name,
                e.description  = $description
            MERGE (s)-[r:CONTAINS_ENTRY]->(e)
            SET r.position = $position
        """, schedule_id=schedule_id, entry_id=entry_id,
            entry_number=entry.entry_number, arabic_name=entry.arabic_name,
            english_name=entry.english_name, description=entry.description,
            position=position)
        if entry.chemical_name:
            self._run("""
                MATCH (e:ScheduleEntry {entry_id: $entry_id})
                MERGE (sub:Substance {substance_id: $sub_id})
                SET sub.arabic_name   = $arabic_name,
                    sub.english_name  = $english_name,
                    sub.chemical_name = $chemical_name,
                    sub.trade_names   = $trade_names
                MERGE (e)-[:DEFINES_SUBSTANCE]->(sub)
            """, entry_id=entry_id,
                sub_id=_stable_id("substance", entry.arabic_name or entry.english_name),
                arabic_name=entry.arabic_name, english_name=entry.english_name,
                chemical_name=entry.chemical_name, trade_names=entry.trade_names or [])
 
    def import_amendment(self, law_id: str, amendment: Amendment):
        """
        Create Amendment node and all article-version relationships.
 
        modification / deletion:
            existing Article -[:AMENDED_BY]-> Amendment
            new versioned Article -[:SUPERSEDES]-> old Article
 
        addition:
            new Article -[:AMENDED_BY]-> Amendment
            new Article -[:SUPERSEDES]-> old Article (if one exists)
        """
        desc = (amendment.description or "")[:500]
        self._run("""
            MATCH (l:Law {law_id: $law_id})
            MERGE (am:Amendment {amendment_id: $amendment_id})
            SET am.amendment_law_number = $law_num,
                am.amendment_law_title  = $law_title,
                am.amendment_date       = $date,
                am.amendment_type       = $atype,
                am.description          = $desc,
                am.effective_date       = $effective_date
            MERGE (l)-[:HAS_AMENDMENT]->(am)
        """, law_id=law_id, amendment_id=amendment.amendment_id,
            law_num=amendment.amendment_law_number,
            law_title=amendment.amendment_law_title,
            date=amendment.amendment_date, atype=amendment.amendment_type,
            desc=desc, effective_date=amendment.effective_date)
 
        for article_number in amendment.amended_article_numbers:
            if not article_number:
                continue
            is_addition = amendment.amendment_type == "addition"
            tag         = "added" if is_addition else "amended"
            new_id      = _stable_id(law_id, article_number, tag, amendment.amendment_date)
 
            if is_addition:
                self._run("""
                    MATCH (l:Law {law_id: $law_id})
                    MATCH (am:Amendment {amendment_id: $amendment_id})
                    MERGE (a_new:Article {article_id: $new_id})
                    ON CREATE SET
                        a_new.article_number = $article_number,
                        a_new.law_id         = $law_id,
                        a_new.text           = $desc,
                        a_new.version        = $date,
                        a_new.is_addition    = true
                    MERGE (l)-[:CONTAINS]->(a_new)
                    MERGE (a_new)-[:AMENDED_BY {
                        amendment_type: $atype, amendment_date: $date
                    }]->(am)
                """, law_id=law_id, amendment_id=amendment.amendment_id,
                    new_id=new_id, article_number=article_number, desc=desc,
                    date=amendment.amendment_date, atype=amendment.amendment_type)
                self._run("""
                    MATCH (a_new:Article {article_id: $new_id})
                    MATCH (a_old:Article {law_id: $law_id, article_number: $article_number})
                    WHERE a_old.article_id <> $new_id
                      AND a_old.is_addition IS NULL
                    MERGE (a_new)-[:SUPERSEDES {
                        amendment_id: $amendment_id, amendment_date: $date
                    }]->(a_old)
                """, new_id=new_id, law_id=law_id, article_number=article_number,
                    amendment_id=amendment.amendment_id, date=amendment.amendment_date)
            else:
                self._run("""
                    MATCH (am:Amendment {amendment_id: $amendment_id})
                    MATCH (a:Article {law_id: $law_id, article_number: $article_number})
                    MERGE (a)-[r:AMENDED_BY]->(am)
                    SET r.amendment_type = $atype,
                        r.amendment_date = $date
                """, amendment_id=amendment.amendment_id, law_id=law_id,
                    article_number=article_number,
                    atype=amendment.amendment_type, date=amendment.amendment_date)
                self._run("""
                    MATCH (l:Law {law_id: $law_id})
                    MATCH (am:Amendment {amendment_id: $amendment_id})
                    MATCH (a_old:Article {law_id: $law_id, article_number: $article_number})
                    WHERE a_old.article_id <> $new_id
                      AND a_old.is_amended IS NULL
                    MERGE (a_new:Article {article_id: $new_id})
                    ON CREATE SET
                        a_new.article_number = $article_number,
                        a_new.law_id         = $law_id,
                        a_new.text           = $desc,
                        a_new.version        = $date,
                        a_new.is_amended     = true
                    MERGE (l)-[:CONTAINS]->(a_new)
                    MERGE (a_new)-[:SUPERSEDES {
                        amendment_id: $amendment_id, amendment_date: $date
                    }]->(a_old)
                """, law_id=law_id, amendment_id=amendment.amendment_id,
                    article_number=article_number, new_id=new_id,
                    desc=desc, date=amendment.amendment_date)
 
    def import_law(self, summary: Dict, verbose: bool = True):
        """
        Import one law from an ``ingest_dataset()`` summary dict.
        All structured data is pre-extracted — no re-parsing.
        NOTE: schedules are intentionally empty here; they are imported
        separately in Phase 4 from dedicated table chunks.
        """
        law_id = summary["law_meta"]["law_id"]
        if verbose:
            print(f"\n{'='*60}\nIMPORTING: {summary['law_meta'].get('title')}\n{'='*60}")
        self.create_law(summary["law_meta"])
        for i, a in enumerate(summary["articles"]):
            self.create_article(law_id, a.get("article_number"), a.get("text", ""), i)
        for p in summary["penalties"]:
            self.create_penalty(p, law_id)
        for d in summary["definitions"]:
            self.create_definition(d, law_id)
        for topic_name in set(t["topic_name"] for t in summary["topics"]):
            self.create_topic(topic_name)
        for t in summary["topics"]:
            self.link_article_topic(law_id, t["article_number"],
                                    t["topic_name"], t.get("confidence", 1.0))
        for r in summary["references"]:
            self.create_reference(law_id, r["from_article"], r["to_article"])
        if verbose:
            print(f"  ✓ {len(summary['articles'])} articles "
                  f"| {len(summary['penalties'])} penalties "
                  f"| {len(summary['schedules'])} schedules")
 
    # ── statistics / queries ──────────────────────────────────────────────
 
    def get_statistics(self) -> Dict[str, Any]:
        stats: Dict[str, Any] = {"nodes": {}, "relationships": {}}
        with self.driver.session() as s:
            for label in norm_regu.NODE_LABELS.value:
                stats["nodes"][label] = s.run(
                    f"MATCH (n:{label}) RETURN count(n) AS c").single()["c"]
            for rel in norm_regu.RELATIONSHIPS.value:
                stats["relationships"][rel] = s.run(
                    f"MATCH ()-[r:{rel}]->() RETURN count(r) AS c").single()["c"]
        return stats
 
    def print_statistics(self):
        stats = self.get_statistics()
        print("\n" + "="*60 + "\nKNOWLEDGE GRAPH STATISTICS\n" + "="*60)
        print("\nNodes:")
        for k, v in stats["nodes"].items():
            print(f"  {k}: {v}")
        print("\nRelationships:")
        for k, v in stats["relationships"].items():
            print(f"  {k}: {v}")
        print("="*60)
 
    def query_amendments(self, law_id: Optional[str] = None) -> List[Dict]:
        with self.driver.session() as s:
            if law_id:
                result = s.run("""
                    MATCH (l:Law {law_id: $law_id})-[:HAS_AMENDMENT]->(am:Amendment)
                    OPTIONAL MATCH (a:Article)-[:AMENDED_BY]->(am)
                    RETURN am, collect(DISTINCT a.article_number) AS affected_articles
                    ORDER BY am.amendment_date
                """, law_id=law_id)
            else:
                result = s.run("""
                    MATCH (l:Law)-[:HAS_AMENDMENT]->(am:Amendment)
                    OPTIONAL MATCH (a:Article)-[:AMENDED_BY]->(am)
                    RETURN l.law_id AS law_id, l.title AS law_title,
                           am, collect(DISTINCT a.article_number) AS affected_articles
                    ORDER BY am.amendment_date
                """)
            out = []
            for record in result:
                row = dict(record["am"])
                row["affected_articles"] = record["affected_articles"]
                if not law_id:
                    row["law_id"]    = record["law_id"]
                    row["law_title"] = record["law_title"]
                out.append(row)
            return out
 
    def query_article_history(self, law_id: str, article_number: str) -> List[Dict]:
        with self.driver.session() as s:
            result = s.run("""
                MATCH (a:Article {law_id: $law_id, article_number: $article_number})
                OPTIONAL MATCH (a)-[:AMENDED_BY]->(am:Amendment)
                OPTIONAL MATCH (a_new:Article)-[:SUPERSEDES]->(a)
                OPTIONAL MATCH (a)-[:SUPERSEDES]->(a_old:Article)
                RETURN
                    a.article_number                   AS article,
                    a.version                          AS version,
                    a.is_amended                       AS is_amended,
                    a.is_addition                      AS is_addition,
                    collect(DISTINCT {
                        amendment_id: am.amendment_id,
                        law_num:      am.amendment_law_number,
                        date:         am.amendment_date,
                        type:         am.amendment_type
                    })                                 AS amendments,
                    collect(DISTINCT a_new.article_id) AS superseded_by,
                    collect(DISTINCT a_old.article_id) AS supersedes
                ORDER BY a.version
            """, law_id=law_id, article_number=article_number)
            return [dict(r) for r in result]

In [5]:
# =============================================================================
# SECTION 6: PIPELINE
# =============================================================================
 
def build_knowledge_graph(
    neo4j_uri:      str,
    neo4j_user:     str,
    neo4j_password: str,
    drop_existing:  bool = True,
    verbose:        bool = True,
) -> LegalKnowledgeGraph:
    """
    Full pipeline. Calls ``get_chunks()`` once internally.
 
    Phase 0 — chunk all law files.
    Phase 1 — extract structured data from article chunks.
    Phase 2 — import laws + articles + NLP data into Neo4j.
    Phase 3 — import amendments from amended_article chunks.
    Phase 4 — import schedules/tables from table chunks.
    """
    # ── Phase 0 ───────────────────────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE 0: CHUNKING\n" + "="*60)
    chunks = get_chunks()
    print(f"  Total chunks: {len(chunks):,}")
    chunk_type_counts = {}
    for c in chunks:
        ct = c.metadata.get("chunk_type", "unknown")
        chunk_type_counts[ct] = chunk_type_counts.get(ct, 0) + 1
    for ct, count in sorted(chunk_type_counts.items()):
        print(f"  {ct}: {count}")
 
    # ── Phase 1 ───────────────────────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE 1: EXTRACTION SUMMARY\n" + "="*60)
    summaries  = ingest_dataset(chunks)
    _LIST_KEYS = {"articles", "penalties", "definitions", "references",
                  "topics", "schedules", "law_meta"}
    df_rows = []
    for s in summaries:
        row = {k: v for k, v in s.items() if k not in _LIST_KEYS}
        for k in ("articles", "penalties", "definitions", "references", "topics"):
            row[k] = len(s.get(k) or [])
        df_rows.append(row)
    print(pd.DataFrame(df_rows).to_string())
    successful = [s for s in summaries if not s.get("error")]
    print(f"\n  ✓ {len(successful)}/{len(summaries)} laws extracted successfully")
 
    # ── Phase 2 ───────────────────────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE 2: IMPORTING LAWS\n" + "="*60)
    graph = LegalKnowledgeGraph(neo4j_uri, neo4j_user, neo4j_password)
    graph.setup_schema(drop_existing=drop_existing)
    for summary in successful:
        graph.import_law(summary, verbose=verbose)
 
    # ── Phase 3 ───────────────────────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE 3: IMPORTING AMENDMENTS\n" + "="*60)
    amendments_by_law = ingest_amendments(chunks)
    total_amendments = 0
    for law_id, amendments in amendments_by_law.items():
        if verbose:
            print(f"\n  → {law_id}: {len(amendments)} amendment(s)")
        for amendment in amendments:
            try:
                graph.import_amendment(law_id, amendment)
                total_amendments += 1
                if verbose:
                    print(f"    ✓ law={amendment.amendment_law_number}/"
                          f"{amendment.amendment_date[:4]} "
                          f"| type={amendment.amendment_type} "
                          f"| articles={amendment.amended_article_numbers}")
            except Exception as e:
                logger.error(f"    ✗ {amendment.amendment_id}: {e}")
                if verbose:
                    traceback.print_exc()
    print(f"\n  ✓ Total amendments imported: {total_amendments}")
 
    # ── Phase 4 ───────────────────────────────────────────────────────────
    print("\n" + "="*60 + "\nPHASE 4: IMPORTING TABLES\n" + "="*60)
    tables_by_law = ingest_tables(chunks)
    total_tables, total_entries = 0, 0
    for law_id, schedules in tables_by_law.items():
        if verbose:
            print(f"\n  → {law_id}: {len(schedules)} table(s)")
        for sched in schedules:
            try:
                graph.create_schedule(law_id, sched)
                for i, entry in enumerate(sched.entries):
                    graph.create_schedule_entry(sched.schedule_id, entry, i)
                total_tables  += 1
                total_entries += len(sched.entries)
                if verbose:
                    print(f"    ✓ table={sched.schedule_number} "
                          f"| {len(sched.entries)} entries "
                          f"| {sched.title[:50]}")
            except Exception as e:
                logger.error(f"    ✗ table {sched.schedule_number} "
                             f"(law={law_id}): {e}")
                if verbose:
                    traceback.print_exc()
    print(f"\n  ✓ Tables imported: {total_tables} | Entries: {total_entries}")
 
    print("\n✅ BUILD COMPLETE")
    graph.print_statistics()
    return graph

In [6]:
# =============================================================================
# RUN
# =============================================================================
 
graph = build_knowledge_graph(
    neo4j_uri      = get_settings().NEO4J_URI,
    neo4j_user     = get_settings().NEO4J_USERNAME,
    neo4j_password = get_settings().NEO4J_PASSWORD,
    drop_existing  = True,
    verbose        = True,
)
graph.close()

2026-03-13 03:03:54,626 - INFO -   3okobat.txt [article] → 538 chunks
2026-03-13 03:03:54,627 - INFO -   new_3okobat_num36_2020.txt [amendment] → 1 chunks
2026-03-13 03:03:54,627 - INFO -   new_3okobat_num6_2020.txt [amendment] → 2 chunks
2026-03-13 03:03:54,628 - INFO -   قانون-مكافحة-غسل-الامول_ultra_clean.txt [article] → 21 chunks
2026-03-13 03:03:54,629 - INFO -   asle7a.txt [article] → 52 chunks
2026-03-13 03:03:54,629 - INFO -   asle7a_tables.txt [table] → 1 chunks
2026-03-13 03:03:54,632 - INFO -   dostor_masr.txt [article] → 252 chunks
2026-03-13 03:03:54,633 - INFO -   drugs.txt [article] → 69 chunks
2026-03-13 03:03:54,633 - INFO -   drugs_tables.txt [table] → 1 chunks
2026-03-13 03:03:54,637 - INFO -   egra2at_gena2ya.txt [article] → 511 chunks
2026-03-13 03:03:54,638 - INFO -   erhab.txt [article] → 55 chunks
2026-03-13 03:03:54,639 - INFO -   قانون-الطوارئ_ultra_clean.txt [article] → 21 chunks
2026-03-13 03:03:54,640 - INFO -   tech.txt [article] → 46 chunks
2026-03-13 03:


PHASE 0: CHUNKING
  Total chunks: 1,570
  amended_article: 3
  article: 1557
  preamble: 8
  table: 2

PHASE 1: EXTRACTION SUMMARY
                 law_key error  articles  penalties  definitions  references  topics
0             penal_code  None       537         65            1         103      81
1       money_laundering  None        20          0            0          12       1
2     weapons_ammunition  None        51          0            0          11      16
3  criminal_constitution  None       251          0            0          15       2
4             anti_drugs  None        68          1            0          26      40
5     criminal_procedure  None       510         16            0          86       9
6            anti_terror  None        54          0            0          18       4
7          emergency_law  None        21          2            0           1       2
8             cybercrime  None        45          0            1           9       0

  ✓ 9/9 laws extr

2026-03-13 03:03:59,632 - INFO - ✓ Neo4j connected
2026-03-13 03:03:59,845 - INFO - ✓ Existing data cleared
2026-03-13 03:04:00,935 - INFO - Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT law_id_unique IF NOT EXISTS FOR (e:Law) REQUIRE (e.law_id) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT constraint_b9269926 FOR (e:Law) REQUIRE (e.law_id) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT law_id_unique        IF NOT EXISTS FOR (l:Law)          REQUIRE l.law_id IS UNIQUE'
202


IMPORTING: قانون العقوبات
  ✓ 537 articles | 65 penalties | 0 schedules

IMPORTING: قانون مكافحة غسل الأموال
  ✓ 20 articles | 0 penalties | 0 schedules

IMPORTING: قانون الأسلحة والذخيرة
  ✓ 51 articles | 0 penalties | 0 schedules

IMPORTING: الدستور الجنائي
  ✓ 251 articles | 0 penalties | 0 schedules

IMPORTING: قانون مكافحة المخدرات
  ✓ 68 articles | 1 penalties | 0 schedules

IMPORTING: قانون الإجراءات الجنائية
  ✓ 510 articles | 16 penalties | 0 schedules

IMPORTING: قانون مكافحة الإرهاب
  ✓ 54 articles | 0 penalties | 0 schedules

IMPORTING: قانون الطوارئ
  ✓ 21 articles | 2 penalties | 0 schedules

IMPORTING: قانون مكافحة جرائم تقنية المعلومات
  ✓ 45 articles | 0 penalties | 0 schedules

PHASE 3: IMPORTING AMENDMENTS

  → penal_code: 2 amendment(s)
    ✓ law=58/2020 | type=addition | articles=['309/ب']


2026-03-13 03:15:59,262 - INFO -   ✓ weapons_ammunition: 5 table(s) | table 1 (8 entries) | table 2 (0 entries) | table 3 (0 entries) | table 4 (15 entries) | table 5 (0 entries)
2026-03-13 03:15:59,263 - INFO -   ✓ anti_drugs: 10 table(s) | table 1 (0 entries) | table 2 (1 entries) | table 3 (85 entries) | table 4 (0 entries) | table 4 (0 entries) | table 4 (0 entries) | table 4 (0 entries) | table 5 (0 entries) | table 5 (0 entries) | table 6 (0 entries)


    ✓ law=58/2020 | type=modification | articles=['392']

  ✓ Total amendments imported: 2

PHASE 4: IMPORTING TABLES

  → weapons_ammunition: 5 table(s)
    ✓ table=1 | 8 entries | بيان الأسلحة البيضاء
    ✓ table=2 | 0 entries | الأسلحة النارية غير المششخنة
    ✓ table=3 | 0 entries | الأسلحة المششخنة (2)
    ✓ table=4 | 15 entries | الأجزاء الرئيسية للأسلحة النارية
    ✓ table=5 | 0 entries | (4)

  → anti_drugs: 10 table(s)
    ✓ table=1 | 0 entries | المواد المعتبرة مخدرة
    ✓ table=2 | 1 entries | المستحضرات المستثناة من النظام المطبق على المواد ا
    ✓ table=3 | 85 entries | في المواد التي تخضع لبعض قيود الجواهر المخدرة
    ✓ table=4 | 0 entries | الحد الأقصى لكميات الجواهر المخدرة الذي لا يجوز ل
    ✓ table=4 | 0 entries | الملحق بالقانون الجواهر المخدرة الاتية وفقاً لما ج
    ✓ table=4 | 0 entries | تحت رقم (28) وفقاً لما جاء بقرار وزير الصحة رقم 16
    ✓ table=4 | 0 entries | وفقاً لما جاء بقرار وزير الصحة رقم 46 لسنة 1997 ال
    ✓ table=5 | 0 entries | النباتات الممن

2026-03-13 03:16:46,886 - WARNING - Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `DEFINES_SUBSTANCE` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:DEFINES_SUBSTANCE]->() RETURN count(r) AS c'



KNOWLEDGE GRAPH STATISTICS

Nodes:
  Law: 9
  Article: 1559
  Penalty: 84
  Definition: 2
  Topic: 5
  Schedule: 11
  ScheduleEntry: 109
  Substance: 0
  Amendment: 1

Relationships:
  CONTAINS: 1559
  REFERENCES: 459
  HAS_PENALTY: 136
  DEFINES: 2
  TAGGED_WITH: 212
  HAS_SCHEDULE: 11
  CONTAINS_ENTRY: 109
  DEFINES_SUBSTANCE: 0
  HAS_AMENDMENT: 1
  AMENDED_BY: 2
  SUPERSEDES: 1
